# 10 - SHAP feature matrix (one row per company, all variables)

Assembles the modelling table for the SHAP model by combining, on `CompanyNumber`:

1. **Raw** Companies House columns (passthrough) from `01_companies_house`.
2. **Static engineered** features for **all** 1.37M companies (`src.features.ch_static`).
3. **API-derived** features for a prioritised **sample** (`src.features.ch_api`, NB05 contracts);
   sentinel elsewhere, flagged by `api_enriched`.

Every variable is documented in `reports/shap_feature_catalog.md`. `CompanyNumber` is kept
**verbatim** (never zero-padded) so other databases can be joined later; an 8-digit
`_join_key` is used internally for API merges and dropped before writing.

In [ ]:
# --- Config ---
import os
N = int(os.getenv("FEATURE_MATRIX_N", "1000"))   # live CH-API sample size (knob: scale up when ready)
RUN_CH_API = True # fetch officers / filings / charges / lender for the sample (needs CH_API_KEY)
API_SLEEP = 0.5 # seconds between companies (rate limiting)
CONTRACTS_CSV = None # path to an NB05 company-level contracts CSV to merge; None -> sentinel

BASE_CSV        = "../data/processed/filtered_bb_sme_sectors.csv"
OUT_PARQUET     = "../data/processed/shap_feature_matrix.parquet" # I am saving as .parquet because it preserves dtypes
OUT_CATALOG_CSV = "../data/processed/shap_feature_catalog.csv"


In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
sys.path.insert(0, str(Path.cwd().parent))   # repo root on path -> `import src...`
from src.features import ch_static, ch_api, sample

print("CH API key present:", ch_api.has_api_key())

CH API key present: True


## 1. Load base data (CompanyNumber kept verbatim)

In [3]:
# Read everything as string so CompanyNumber (which may have leading zeros / alpha prefixes)
# is preserved byte-for-byte. ch_static coerces the numeric columns it needs.
df = pd.read_csv(BASE_CSV, dtype=str, low_memory=False)
ch_static.strip_columns(df)

orig_company_number = df["CompanyNumber"].copy()          # for the verbatim assertion later
df[sample.JOIN_KEY] = df["CompanyNumber"].str.strip().str.upper().str.zfill(8)

print("Base shape:", df.shape)
print("CompanyNumber unique:", df["CompanyNumber"].is_unique)
df[["CompanyName", "CompanyNumber", "sector", "segment"]].head()

Base shape: (1372321, 58)
CompanyNumber unique: True


,CompanyName,CompanyNumber,sector,segment
0,!ABRIDGE TAX LTD,16092999,Fast growth & emerging,No Filings
1,!BIG IMPACT GRAPHICS LIMITED,11743365,"Technology, legal & professional",Dormant
2,!NFLECTION ADVISORY LIMITED,15073164,"Technology, legal & professional",Small
3,!NFOGENIE LTD,13522064,"Technology, legal & professional",Micro
4,!NNOV8 LIMITED,11006939,Fast growth & emerging,Micro


In [4]:
df

,CompanyName,CompanyNumber,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,PreviousName_8.CompanyName,PreviousName_9.CONDATE,PreviousName_9.CompanyName,PreviousName_10.CONDATE,PreviousName_10.CompanyName,ConfStmtNextDueDate,ConfStmtLastMadeUpDate,sector,segment,_join_key
0,!ABRIDGE TAX LTD,16092999,NaN,NaN,82 GREAT NORTH ROAD,GREAT NORTH BUSINESS CENTRE,HATFIELD,NaN,UNITED KINGDOM,AL9 5BL,...,NaN,NaN,NaN,NaN,NaN,04/12/2026,20/11/2025,Fast growth & emerging,No Filings,16092999
1,!BIG IMPACT GRAPHICS LIMITED,11743365,NaN,NaN,124 CITY ROAD,NaN,LONDON,NaN,NaN,EC1V 2NX,...,NaN,NaN,NaN,NaN,NaN,29/12/2026,15/12/2025,"Technology, legal & professional",Dormant,11743365
2,!NFLECTION ADVISORY LIMITED,15073164,NaN,NaN,74 SANTERS LANE,NaN,POTTERS BAR,HERTFORDSHIRE,ENGLAND,EN6 2DA,...,NaN,NaN,NaN,NaN,NaN,28/08/2026,14/08/2025,"Technology, legal & professional",Small,15073164
3,!NFOGENIE LTD,13522064,NaN,NaN,71-75 SHELTON STREET,NaN,LONDON,GREATER LONDON,UNITED KINGDOM,WC2H 9JQ,...,NaN,NaN,NaN,NaN,NaN,03/08/2026,20/07/2025,"Technology, legal & professional",Micro,13522064
4,!NNOV8 LIMITED,11006939,NaN,NaN,OLD BARN FARM,HARTFIELD ROAD,EDENBRIDGE,NaN,ENGLAND,TN8 5NF,...,NaN,NaN,NaN,NaN,NaN,24/10/2026,10/10/2025,Fast growth & emerging,Micro,11006939
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1372316,ŚŪNYA NFP CIC,16780614,NaN,NaN,17 CHAMBERLAIN CLOSE,NaN,ILFORD,NaN,ENGLAND,IG1 1JQ,...,NaN,NaN,NaN,NaN,NaN,26/10/2026,NaN,"Technology, legal & professional",No Filings,16780614
1372317,ŠIŠARKA LTD,16903765,NaN,NaN,71-75 SHELTON STREET,COVENT GARDEN,LONDON,NaN,UNITED KINGDOM,WC2H 9JQ,...,NaN,NaN,NaN,NaN,NaN,24/12/2026,NaN,"Technology, legal & professional",No Filings,16903765
1372318,ŪRVY ELECTRIC LIMITED,16505142,NaN,NaN,167 - 169 GREAT PORTLAND STREET,5TH FLOOR,LONDON,NaN,ENGLAND,W1W 5PF,...,NaN,NaN,NaN,NaN,NaN,22/06/2026,NaN,Manufacturing,No Filings,16505142
1372319,‘LIVE THE BUZZ’ C.I.C.,13733947,NaN,NaN,'LIVE THE BUZZ' C.I.C 3RD FLOOR,86-90 PAUL STREET,LONDON,NaN,ENGLAND,EC2A 4NE,...,NaN,NaN,NaN,NaN,NaN,22/11/2026,08/11/2025,"Technology, legal & professional",Small,13733947


## 2. Static engineered features (all companies)

In [4]:
ch_static.add_static_features(df)
df[ch_static.STATIC_FEATURE_COLS].describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
company_age_years,1372321.0,NaN,NaN,NaN,9.551067,11.084623,0.1,2.1,6.2,13.2,169.8
has_any_charge,1372321,2,False,1243684,NaN,NaN,NaN,NaN,NaN,NaN,NaN
has_outstanding_charges,1372321,2,False,1279981,NaN,NaN,NaN,NaN,NaN,NaN,NaN
debt_ratio,1372321.0,NaN,NaN,NaN,0.051579,0.207928,0.0,0.0,0.0,0.0,1.0
accounts_overdue,1372321,2,False,1294199,NaN,NaN,NaN,NaN,NaN,NaN,NaN
accounts_stale,1372321,2,False,1049855,NaN,NaN,NaN,NaN,NaN,NaN,NaN
size_tier,899942,4,Micro,501253,NaN,NaN,NaN,NaN,NaN,NaN,NaN
num_previous_names,1372321.0,NaN,NaN,NaN,0.134462,0.430646,0.0,0.0,0.0,0.0,10.0
has_name_change,1372321,2,False,1223103,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Build the prioritised API sample (size = N)

In [5]:
sub = sample.build_api_sample(df, n=N)
print("Companies selected for live CH API:", len(sub))
sub.head()

Companies selected for live CH API: 200


,CompanyName,CompanyNumber,_join_key
0,BONHURST LIMITED,SC106917,SC106917
1,BONI LTD,06439134,06439134
2,BONIFACE ENGINEERING LIMITED,02183008,02183008
3,NG1 LIMITED,03858433,03858433
4,NG15 LTD,04213449,04213449


## 4. Live Companies House API enrichment (officers / filings / charges / lender)

In [6]:
if RUN_CH_API and ch_api.has_api_key():
    api_df = ch_api.enrich_companies(sub, number_col=sample.JOIN_KEY, sleep=API_SLEEP)
    enriched_keys = set(api_df[sample.JOIN_KEY])
    print("Enriched", len(api_df), "companies via CH API.")
else:
    print("Skipping CH API (no key or RUN_CH_API=False) -> CH-API features will be sentinel.")
    api_df = pd.DataFrame(columns=[sample.JOIN_KEY, *ch_api.API_FEATURE_SENTINELS])
    enriched_keys = set()

df["api_enriched"] = df[sample.JOIN_KEY].isin(enriched_keys)
df = df.merge(api_df, on=sample.JOIN_KEY, how="left")

# Sentinel-fill every CH-API column for the non-sampled companies.
for col, sentinel in ch_api.API_FEATURE_SENTINELS.items():
    if col not in df.columns:
        df[col] = sentinel
    if sentinel is not pd.NA and sentinel is not pd.NaT:
        df[col] = df[col].fillna(sentinel)

print("api_enriched rows:", int(df["api_enriched"].sum()))

  enriched 25/200


  enriched 50/200


  enriched 75/200


  enriched 100/200


  enriched 125/200


  enriched 150/200


  enriched 175/200


  enriched 200/200


Enriched 200 companies via CH API.


api_enriched rows: 200


## 5. Contracts Finder features (public API; sample = companies in award notices)

In [7]:
CONTRACT_SENTINELS = {
    "contracts_won_as_supplier": 0,
    "total_value_won_gbp": 0.0,
    "avg_value_won_gbp": 0.0,
    "contracts_given_as_buyer": 0,
    "total_value_given_gbp": 0.0,
    "total_contracts_linked": 0,
    "total_value_linked_gbp": 0.0,
    "latest_supplier_award_date": pd.NaT,
}

if CONTRACTS_CSV:
    c = pd.read_csv(CONTRACTS_CSV, dtype={"CompanyNumber": str})
    c[sample.JOIN_KEY] = c["CompanyNumber"].str.strip().str.upper().str.zfill(8)
    keep = [sample.JOIN_KEY, *[k for k in CONTRACT_SENTINELS if k in c.columns]]
    df = df.merge(c[keep], on=sample.JOIN_KEY, how="left")
    print("Merged contracts for", c[sample.JOIN_KEY].nunique(), "companies.")
else:
    print("No CONTRACTS_CSV set -> contract features will be sentinel (run NB05 to populate).")

for col, sentinel in CONTRACT_SENTINELS.items():
    if col not in df.columns:
        df[col] = sentinel
    if sentinel is not pd.NaT:
        df[col] = df[col].fillna(sentinel)

No CONTRACTS_CSV set -> contract features will be sentinel (run NB05 to populate).


## 6. Composite: financial health concern
`charges_outstanding > 0` (API, sample only) **or** `accounts_overdue` **or** `accounts_stale`
(static, all companies). Computed for every company from whatever signal is available.

In [8]:
df["financial_health_concern"] = (
    (pd.to_numeric(df["charges_outstanding"], errors="coerce").fillna(0) > 0)
    | df["accounts_overdue"].fillna(False)
    | df["accounts_stale"].fillna(True)
)
print(df["financial_health_concern"].value_counts())

financial_health_concern
False    996870
True     375451
Name: count, dtype: int64


## 7. Finalise, validate, write

In [9]:
df = df.drop(columns=[sample.JOIN_KEY])

# --- assertions ---
assert len(df) == 1_372_321, f"row count changed: {len(df)}"
assert df["CompanyNumber"].reset_index(drop=True).equals(
    orig_company_number.reset_index(drop=True)
), "CompanyNumber was mutated (must stay verbatim)"
assert df["CompanyNumber"].is_unique, "CompanyNumber not unique"
assert sample.JOIN_KEY not in df.columns

df.to_parquet(OUT_PARQUET, index=False)
print("Wrote", OUT_PARQUET, "->", df.shape)

Wrote ../data/processed/shap_feature_matrix.parquet -> (1372321, 91)


## 8. Catalog with live coverage + coverage report

In [10]:
ENGINEERED = set(ch_static.STATIC_FEATURE_COLS)
CH_API = set(ch_api.API_FEATURE_SENTINELS) | {"financial_health_concern", "api_enriched"}
CONTRACTS = set(CONTRACT_SENTINELS)

def layer(col):
    if col in ENGINEERED:  return "static_engineered"
    if col in CH_API:      return "api_ch"
    if col in CONTRACTS:   return "api_contracts"
    return "raw"

catalog = pd.DataFrame({
    "variable": df.columns,
    "layer": [layer(c) for c in df.columns],
    "dtype": [str(df[c].dtype) for c in df.columns],
    "nonnull_frac": [df[c].notna().mean() for c in df.columns],
})
catalog.to_csv(OUT_CATALOG_CSV, index=False)

print("Total variables:", len(catalog))
print(catalog["layer"].value_counts().to_string())
print("\nCoverage:")
print("  api_enriched companies:", int(df["api_enriched"].sum()))
print("  companies with any contract:", int((df["contracts_won_as_supplier"] > 0).sum()))
catalog

Total variables: 91
layer
raw                  57
api_ch               17
static_engineered     9
api_contracts         8

Coverage:
  api_enriched companies: 200
  companies with any contract: 0


,variable,layer,dtype,nonnull_frac
0,CompanyName,raw,str,1.000000
1,CompanyNumber,raw,str,1.000000
2,RegAddress.CareOf,raw,str,0.004490
3,RegAddress.POBox,raw,str,0.001434
4,RegAddress.AddressLine1,raw,str,0.999985
...,...,...,...,...
86,total_value_given_gbp,api_contracts,float64,1.000000
87,total_contracts_linked,api_contracts,int64,1.000000
88,total_value_linked_gbp,api_contracts,float64,1.000000
89,latest_supplier_award_date,api_contracts,datetime64[ns],0.000000
